**Name: Rishi Raj Phadale**  
**Roll No.: 23102C0070**  
**R Programming Experiment 3**  

## **Image Recognition & Classification with Keras in R**

## 1. Setup in Colab

In [3]:
system("sudo apt-get update && sudo apt-get install -y python3-venv python3-pip python3-dev libfftw3-dev")

# EBImage isn't on CRAN, install via Bioconductor
if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("EBImage", update = FALSE, ask = FALSE)

install.packages("keras")
library(keras)
install_keras()  # installs TensorFlow backend into a python env

library(EBImage)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.1 (2026-06-24)

Installing package(s) 'EBImage'

also installing the dependency ‘fftwtools’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Using Python: /usr/bin/python3.10
Creating virtual environment 'r-tensorflow' ... 


+ /usr/bin/python3.10 -m venv /root/.virtualenvs/r-tensorflow



Done!
Installing packages: pip, wheel, setuptools


+ /root/.virtualenvs/r-tensorflow/bin/python -m pip install --upgrade pip wheel setuptools



Virtual environment 'r-tensorflow' successfully created.
Using virtual environment 'r-tensorflow' ...


+ /root/.virtualenvs/r-tensorflow/bin/python -m pip install --upgrade --no-user 'numpy<2' 'tensorflow[and-cuda]==2.15.*' tensorflow-hub tensorflow-datasets scipy requests Pillow h5py pandas pydot tf-keras



creating symlinks:
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libcublasLt.so.12' -> '../nvidia/cublas/lib/libcublasLt.so.12'
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libcublas.so.12' -> '../nvidia/cublas/lib/libcublas.so.12'
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libnvblas.so.12' -> '../nvidia/cublas/lib/libnvblas.so.12'
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libcheckpoint.so' -> '../nvidia/cuda_cupti/lib/libcheckpoint.so'
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libcupti.so.12' -> '../nvidia/cuda_cupti/lib/libcupti.so.12'
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libnvperf_host.so' -> '../nvidia/cuda_cupti/lib/libnvperf_host.so'
- '/root/.virtualenvs/r-tensorflow/lib/python3.10/site-packages/tensorflow/libnvperf_target.so' -> '../nvidia/cuda_cupti/lib/libnvperf_target.so'
- '/root/.v

## 2. Read images

In [4]:
setwd('images')
pics <- c('p1.jpg', 'p2.jpg', 'p3.jpg', 'p4.jpg', 'p5.jpg', 'p6.jpg',
          'c1.jpg', 'c2.jpg', 'c3.jpg', 'c4.jpg', 'c5.jpg', 'c6.jpg')
mypic <- list()
for (i in 1:12) {mypic[[i]] <- readImage(pics[i])}

## 3. Resize & reshape

In [5]:
for (i in 1:12) {mypic[[i]] <- resize(mypic[[i]], 28, 28)}
for (i in 1:12) {mypic[[i]] <- array_reshape(mypic[[i]], c(28, 28, 3))}

## 4. Train/test split

In [6]:
trainx <- NULL
for (i in c(1:5, 7:11)) {trainx <- rbind(trainx, mypic[[i]])}
str(trainx)

testx <- rbind(mypic[[6]], mypic[[12]])

trainy <- c(0,0,0,0,0,1,1,1,1,1)
testy  <- c(0, 1)

 num [1:10, 1:2352] 1 1 1 0.26 0.49 ...


## 5. One-hot encoding

In [7]:
trainLabels <- to_categorical(trainy)
testLabels  <- to_categorical(testy)

## 6. Model

In [8]:
model <- keras_model_sequential()
model %>%
  layer_dense(units = 256, activation = 'relu', input_shape = c(2352)) %>%
  layer_dense(units = 128, activation = 'relu') %>%
  layer_dense(units = 2, activation = 'softmax')
summary(model)

Model: "sequential"
________________________________________________________________________________
 Layer (type)                       Output Shape                    Param #     
 dense_2 (Dense)                    (None, 256)                     602368      
 dense_1 (Dense)                    (None, 128)                     32896       
 dense (Dense)                      (None, 2)                       258         
Total params: 635522 (2.42 MB)
Trainable params: 635522 (2.42 MB)
Non-trainable params: 0 (0.00 Byte)
________________________________________________________________________________


## 7. Compile

In [9]:
model %>%
  compile(loss = 'categorical_crossentropy',
          optimizer = optimizer_rmsprop(),
          metrics = c('accuracy'))

## 8. Fit

In [10]:
history <- model %>%
  fit(trainx,
      trainLabels,
      epochs = 30,
      batch_size = 32,
      validation_split = 0.2)

## 9. Evaluate & Predict

In [11]:
model %>% evaluate(trainx, trainLabels)

prob <- model %>% predict(trainx)          # matrix of class probabilities
pred <- apply(prob, 1, which.max) - 1      # convert to class labels (0/1)

table(Predicted = pred, Actual = trainy)

cbind(prob, Predicted = pred, Actual = trainy)

loss  accuracy 
0.2403556 0.9000000

         Actual
Predicted 0 1
        0 5 1
        1 0 4

,,Predicted,Actual
0.998372257,0.001627826,0,0
0.998424768,0.001575250,0,0
0.998336494,0.001663591,0,0
0.996142089,0.003857910,0,0
0.995546341,0.004453649,0,0
0.009690650,0.990309298,1,1
0.001972747,0.998027265,1,1
0.005246521,0.994753480,1,1
0.020102413,0.979897559,1,1
0.904923856,0.095076196,0,1
